# Build `ROQ_basis_2` with `mlgw_bns_jax` (JAX-native, GPU-accelerated)

This notebook builds the **ROQ_basis_2** Reduced Order Quadrature basis for the
`mlgw_bns_jax` gravitational waveform surrogate, reproducing the parameters
described in the paper:

> *"A pre-selected basis is constructed using **Npre = 200 (10)** elements for
> the linear (quadratic) case, and **Nstep = 1000** points at each step.
> We set **three enrichment cycles** each composed of
> **[10⁴, 10⁵, 10⁵] datapoints**, Ni_out = 0 and a respective relative
> tolerance of **[0.1, 1.0, 1.0]**.  The resulting bases are composed of
> **267 (10) linear (quadratic) elements**, achieving a linear (quadratic)
> frequency axis reduction factor of 950 (25 300)."*

**Key features:**
- Runs on **GPU** via `jax[cuda12]` for accelerated waveform generation
- **JIT-compiles** waveform kernels once via `jax.jit` + `jax.vmap`
- **Streaming memory** management — never loads all waveforms at once
- **Checkpoint / resume** — safe to interrupt and restart

**Runtime requirements:**
- Google Colab with a **GPU runtime** (T4, A100, or L4)
- ~15 GB RAM (standard Colab is fine)
- Several hours for the full build (GPU-dependent)

---
⚠️ **Before running:** Go to `Runtime → Change runtime type → GPU`


## 1. Install dependencies

Install JAX with CUDA 12 support **first** (before any JAX import),
then the remaining dependencies.  Order matters to avoid CPU/GPU conflicts.


In [ ]:
# Step 1: Install JAX with CUDA 12 support (must come before any JAX import)
!pip install --upgrade "jax[cuda12]" 2>&1 | tail -3

# Step 2: Remaining dependencies
!pip install numpy scipy h5py 2>&1 | tail -3

print("\n✓ Dependencies installed")


In [ ]:
# Verify GPU is available BEFORE importing the ROQ builder
import os

os.environ["JAX_PLATFORMS"] = "cuda"

import jax
jax.config.update("jax_enable_x64", True)

devices = jax.devices()
print(f"JAX version: {jax.__version__}")
print(f"Devices: {devices}")
print(f"Default backend: {jax.default_backend()}")

if jax.default_backend() != "gpu":
    print("\n⚠️  WARNING: No GPU detected!")
    print("Go to Runtime → Change runtime type → GPU")
    print("Falling back to CPU (will be much slower)...")
    os.environ["JAX_PLATFORMS"] = "cpu"
else:
    gpu = devices[0]
    print(f"\n✓ GPU detected: {gpu}")
    import jax.numpy as jnp
    x = jnp.ones(1000)
    _ = (x @ x).block_until_ready()
    print("✓ GPU compute verified")


## 2. Clone the repository and set up


In [ ]:
import os

REPO_URL = "https://github.com/saulo-albuquerque-phys/mlgw_bns_jax.git"
BRANCH   = "copilot/create-roq-basis-2"  # branch with ROQ_basis_2 scripts
REPO_DIR = "/content/mlgw_bns_jax"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull origin {BRANCH}

os.chdir(REPO_DIR)
print(f"\n✓ Working directory: {os.getcwd()}")
print(f"✓ Branch: {BRANCH}")


In [ ]:
import os

model_path  = "mlgw_bns_jax_model.h5"
config_path = "config_roq_basis_2.ini"
builder_path = "roq_builder_jax.py"

assert os.path.isfile(model_path), (
    f"Model file '{model_path}' not found. "
    "Make sure the repo contains the HDF5 model."
)
print(f"✓ Model file found: {model_path} ({os.path.getsize(model_path)/1e6:.1f} MB)")

assert os.path.isfile(config_path), f"Config file '{config_path}' not found."
print(f"✓ Config file found: {config_path}")

assert os.path.isfile(builder_path), "ROQ builder (roq_builder_jax.py) not found."
print(f"✓ ROQ builder found: {builder_path}")

assert os.path.isfile("build_roq_basis_2.py"), "build_roq_basis_2.py not found."
print("✓ Build script found: build_roq_basis_2.py")


## 3. Configure the ROQ_basis_2 build

We load `config_roq_basis_2.ini` (the paper parameters) and tune the
GPU-specific batch sizes:

| GPU        | `waveform_batch_size` | `projection_batch_size` |
|------------|-----------------------|-------------------------|
| T4  (16 GB)| 3 000                 | 10 000                  |
| L4  (24 GB)| 4 000                 | 15 000                  |
| A100(40 GB)| 6 000                 | 20 000                  |

You can also redirect the output to **Google Drive** by uncommenting
the three lines in the cell below — recommended for long runs.


In [ ]:
import sys
sys.path.insert(0, ".")

from roq_builder_jax import ROQConfig

cfg = ROQConfig.from_ini("config_roq_basis_2.ini")

# ── GPU-optimized batch sizes ────────────────────────────────
if jax.default_backend() == "gpu":
    cfg.waveform_batch_size    = 3000
    cfg.projection_batch_size  = 10000
    print("Using GPU-optimized batch sizes:")
else:
    cfg.waveform_batch_size    = 8
    cfg.projection_batch_size  = 200
    print("Using CPU batch sizes (GPU not available):")
print(f"  waveform_batch_size   = {cfg.waveform_batch_size}")
print(f"  projection_batch_size = {cfg.projection_batch_size}")

# ── (Optional) Save results to Google Drive ───────────────────
# Uncomment the next 3 lines to persist checkpoints to Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# cfg.output_dir = '/content/drive/MyDrive/ROQ_basis_2'

print(f"\nROQ_basis_2 configuration (paper parameters):")
print(f"  Frequency range    : [{cfg.f_min}, {cfg.f_max}] Hz")
print(f"  Segment length     : {cfg.seglen} s  (df = {cfg.delta_f:.4f} Hz)")
print(f"  Frequency points   : {cfg.n_freq}")
print(f"  Pre-basis (lin/qua): {cfg.n_pre_basis_lin} / {cfg.n_pre_basis_qua}  (Npre)")
print(f"  Search iter/step   : {cfg.n_pre_basis_search_iter}  (Nstep)")
print(f"  Enrichment cycles  : {len(cfg.training_set_sizes)}")
for i, (n, rt) in enumerate(zip(cfg.training_set_sizes, cfg.training_set_rel_tol), 1):
    print(f"    Cycle {i}: {n:>7,} waveforms, rel_tol={rt:.1f}")
print(f"  Tolerance (lin)    : {cfg.tolerance_lin:.1e}")
print(f"  Tolerance (qua)    : {cfg.tolerance_qua:.1e}")
print(f"  Output directory   : {cfg.output_dir}")
print(f"  Paper target       : 267 linear nodes (~950x), 10 quadratic (~25300x)")


## 4. JIT warm-up (compile kernels before batch work)

Both the single-waveform and `vmap`-batched kernels are compiled **once**
here.  All subsequent calls reuse the cached XLA code — no Python overhead.
On GPU, compilation typically takes 5–15 seconds.


In [ ]:
import time
import numpy as np
import jax.numpy as jnp

from roq_builder_jax import (
    _load_predictor,
    warmup_jit,
    generate_waveforms_batch,
    normalise,
    sample_parameters,
)

print("Loading waveform model...")
predict_fn = _load_predictor(cfg.model_path)
print("✓ Model loaded")

print(f"\nJIT compiling kernels (batch_size={cfg.waveform_batch_size})...")
t0 = time.time()
precompiled = warmup_jit(
    predict_fn,
    cfg.frequencies,
    cfg.waveform_batch_size,
    verbose=1,
)
print(f"\n✓ Total JIT warm-up: {time.time() - t0:.1f}s")


## 5. Quick benchmark: waveform throughput

After JIT compilation, measure the actual throughput.
The first call above was slow (compilation); subsequent calls are fast.


In [ ]:
rng = np.random.default_rng(42)
test_params = sample_parameters(rng, cfg.waveform_batch_size, cfg)

# Warm call (already compiled)
_ = generate_waveforms_batch(
    predict_fn, test_params, cfg.frequencies,
    batch_size=cfg.waveform_batch_size, _precompiled=precompiled,
)

# Timed call
n_bench = len(test_params)
t0 = time.time()
wf = generate_waveforms_batch(
    predict_fn, test_params, cfg.frequencies,
    batch_size=cfg.waveform_batch_size, _precompiled=precompiled,
)
elapsed = time.time() - t0

print(f"Generated {n_bench} waveforms × {cfg.n_freq} freq points")
print(f"  Total time    : {elapsed:.2f}s")
print(f"  Per waveform  : {elapsed / n_bench * 1000:.1f} ms")
print(f"  Throughput    : {n_bench / elapsed:.1f} waveforms/s")
print(f"  Waveform shape: {wf.shape}")
print(f"  Waveform dtype: {wf.dtype}")

del wf, test_params
import gc; gc.collect()


## 6. Build ROQ_basis_2

The build runs three phases per basis kind:

1. **Pre-selection** (`Npre=200` / `10`, `Nstep=1000`) — greedy scan of
   corners + random waveforms; selects the initial basis vectors.
2. **Enrichment** (3 cycles: 10⁴, 10⁵, 10⁵ waveforms) — streams larger
   training sets and adds any waveform whose projection error exceeds the
   per-cycle effective tolerance (`rel_tol × abs_tol`).
3. **EIM** — selects empirical interpolation nodes from the basis.

Each phase saves a **checkpoint**.  If the runtime disconnects, re-run
this cell — it will **resume** from the last completed phase.

**Expected results (paper):** 267 linear nodes (950× compression),
10 quadratic nodes (25 300× compression).

**Estimated wall-clock time on a T4 GPU:**
- Linear pre-selection (~200 steps × 1 000 candidates): ~1–2 h
- Linear enrichment (2 × 10⁵ waveforms): ~2–4 h
- Quadratic: ~30 min

💡 **Tip:** Mount Google Drive (cell 3) to persist checkpoints across sessions.


In [ ]:
from roq_builder_jax import build_roq_basis
import gc

print("=" * 65)
print("  Building LINEAR ROQ basis  (target: ~267 nodes)")
print("=" * 65)
print(f"  Resume mode: ON (will skip completed phases)")
print()

t0 = time.time()
results_lin = build_roq_basis(cfg, kind="linear", resume=True)
t_lin = time.time() - t0

n_lin = len(results_lin['nodes'])
print(f"\n{'─' * 50}")
print(f"LINEAR basis built in {t_lin:.1f}s ({t_lin/60:.1f} min)")
print(f"  Basis size    : {len(results_lin['basis'])}")
print(f"  Nodes         : {n_lin}  (paper target: 267)")
print(f"  Compression   : {cfg.n_freq / n_lin:.0f}×  (paper target: ~950×)")


In [ ]:
# Mount Google Drive to persist results (recommended for long runs)
# Uncomment and run this cell before starting the quadratic build.

# from google.colab import drive
# drive.mount('/content/drive')
# cfg.output_dir = '/content/drive/MyDrive/ROQ_basis_2'
# print(f"✓ Output redirected to {cfg.output_dir}")


In [ ]:
# Free linear waveform data before the quadratic build
del results_lin["basis"]
gc.collect()

print("=" * 65)
print("  Building QUADRATIC ROQ basis  (target: ~10 nodes)")
print("=" * 65)
print(f"  Resume mode: ON")
print()

t0 = time.time()
results_qua = build_roq_basis(cfg, kind="quadratic", resume=True)
t_qua = time.time() - t0

n_qua = len(results_qua['nodes'])
print(f"\n{'─' * 50}")
print(f"QUADRATIC basis built in {t_qua:.1f}s ({t_qua/60:.1f} min)")
print(f"  Basis size    : {len(results_qua['basis'])}")
print(f"  Nodes         : {n_qua}  (paper target: 10)")
print(f"  Compression   : {cfg.n_freq / n_qua:.0f}×  (paper target: ~25300×)")


## 7. Validate the basis

Generate random unseen waveforms and check that the ROQ reconstruction
error stays below the tolerance.


In [ ]:
from roq_builder_jax import validate_basis
from pathlib import Path

out_lin = Path(cfg.output_dir) / "ROQ_data" / "linear"
out_qua = Path(cfg.output_dir) / "ROQ_data" / "quadratic"

# Reload from disk (safe after resume)
results_lin_disk = {
    "basis":        np.load(out_lin / "basis_linear.npy"),
    "nodes":        np.load(out_lin / "empirical_nodes_linear.npy"),
    "interpolant": np.load(out_lin / "basis_interpolant_linear.npy"),
    "frequencies": cfg.frequencies,
}

# Re-load predictor and warm up for validation
predict_fn_val = _load_predictor(cfg.model_path)
precompiled_val = warmup_jit(
    predict_fn_val, cfg.frequencies, cfg.waveform_batch_size, verbose=0,
)

print("Validating LINEAR basis...")
errors_lin, lin_ok = validate_basis(
    results_lin_disk, predict_fn_val, cfg,
    n_test=200, quadratic=False, precompiled=precompiled_val,
)
status = "✓ PASS" if lin_ok else "✗ FAIL"
print(f"  {status}  max_err={errors_lin.max():.2e}  (tol={cfg.tolerance_lin:.1e})")

if out_qua.exists() and (out_qua / "basis_quadratic.npy").exists():
    results_qua_disk = {
        "basis":        np.load(out_qua / "basis_quadratic.npy"),
        "nodes":        np.load(out_qua / "empirical_nodes_quadratic.npy"),
        "interpolant": np.load(out_qua / "basis_interpolant_quadratic.npy"),
        "frequencies": cfg.frequencies,
    }
    print("\nValidating QUADRATIC basis...")
    errors_qua, qua_ok = validate_basis(
        results_qua_disk, predict_fn_val, cfg,
        n_test=200, quadratic=True, precompiled=precompiled_val,
    )
    status = "✓ PASS" if qua_ok else "✗ FAIL"
    print(f"  {status}  max_err={errors_qua.max():.2e}  (tol={cfg.tolerance_qua:.1e})")
else:
    print("\n(Quadratic basis not yet built — skipping validation)")


## 8. Summary and comparison with paper targets


In [ ]:
import json
from pathlib import Path

out_dir = Path(cfg.output_dir)

print("=" * 65)
print("  ROQ_basis_2 — RESULTS vs PAPER TARGETS")
print("=" * 65)

paper_targets = {
    "linear":    {"nodes": 267, "compression": 950},
    "quadratic": {"nodes": 10,  "compression": 25300},
}

for kind in ["linear", "quadratic"]:
    d = out_dir / "ROQ_data" / kind
    if not d.exists():
        print(f"\n  {kind.upper()}: not built yet")
        continue
    nodes_path = d / f"empirical_nodes_{kind}.npy"
    if not nodes_path.exists():
        print(f"\n  {kind.upper()}: EIM not yet done")
        continue
    n_nodes = len(np.load(nodes_path))
    compression = cfg.n_freq / n_nodes
    tgt = paper_targets[kind]
    print(f"\n  {kind.upper()} basis:")
    print(f"    Nodes        : {n_nodes:4d}  (paper: {tgt['nodes']})")
    print(f"    Compression  : {compression:6.0f}×  (paper: ~{tgt['compression']}×)")

print(f"\n{'─' * 50}")
print("  Output files:")
for kind in ["linear", "quadratic"]:
    d = out_dir / "ROQ_data" / kind
    if not d.exists():
        continue
    print(f"\n  {kind.upper()}/ ({d})")
    for f in sorted(d.iterdir()):
        size = f.stat().st_size
        tag = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
        print(f"    {f.name:50s}  {tag}")

print(f"\n  Full path: {out_dir.resolve()}")
print("\n  Files needed for PE:")
print("    empirical_frequencies_{{linear,quadratic}}.npy")
print("    empirical_nodes_{{linear,quadratic}}.npy")
print("    basis_interpolant_{{linear,quadratic}}.npy")


## 9. (Optional) Save results to Google Drive

If you did not redirect `cfg.output_dir` to Drive earlier, copy the
results now:


In [ ]:
# Uncomment to copy results to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# dest = '/content/drive/MyDrive/ROQ_basis_2'
# shutil.copytree(cfg.output_dir, dest, dirs_exist_ok=True)
# print(f"✓ Results copied to {dest}")


## 10. (Optional) Download key files to local machine

Download only the small files required for ROQ-accelerated PE:


In [ ]:
# Uncomment to download the PE-required files directly from Colab
# from google.colab import files
# from pathlib import Path
#
# for kind in ["linear", "quadratic"]:
#     d = Path(cfg.output_dir) / "ROQ_data" / kind
#     for name in ["empirical_frequencies", "empirical_nodes", "basis_interpolant"]:
#         fpath = d / f"{name}_{kind}.npy"
#         if fpath.exists():
#             files.download(str(fpath))
#             print(f"Downloaded {fpath.name}")
